# 10 — AdaptiveTrend replication (arXiv:2602.11708)

[Bui & Nguyen (2026)](https://arxiv.org/pdf/2602.11708): **6h momentum** + **ATR trailing stop**, monthly Sharpe asset filter, **70/30** long-short.

This notebook runs a **fixed-parameter** subset on Binance Vision (**4h → 6h** resample):

| Paper | This lab |
|---|---|
| 150+ perps, monthly grid per asset | **8 symbols**, fixed `L=20`, `θ=2%`, `α=2.5`, `λ=0.70` |
| OOS Jan 2022 – Dec 2024 | Toggle `paper_window=True` below **or** QMIE OOS 2023→now |
| Claimed Sharpe 2.41, MDD −12.7% | **Replicate to verify** — do not trust without this cell output |

Research only. No live ``W_*``.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif (ROOT / "research").exists():
    pass
elif (ROOT / "python" / "research").exists():
    ROOT = ROOT / "python"
sys.path.insert(0, str(ROOT))
print("python root", ROOT)


In [ ]:
from research.trend_lab.adaptivetrend import AdaptiveTrendParams, eval_adaptivetrend_universe
from research.trend_lab.data import CORE
from research.trend_lab.run_adaptivetrend_validation import DEFAULT_SYMBOLS

p = AdaptiveTrendParams()
symbols = DEFAULT_SYMBOLS
print("params", p)
print("symbols", symbols)


In [ ]:
# Paper-aligned OOS window (2022-01-01 .. 2024-12-31)
paper = eval_adaptivetrend_universe(symbols, p, paper_window=True)
print("Paper window OOS:", paper["oos"])
print("Long book:", paper["oos_long"])
print("Short book:", paper["oos_short"])
paper["per_symbol"]


In [ ]:
# QMIE protocol OOS (2023 → today) for desk comparability
qmie = eval_adaptivetrend_universe(symbols, p, paper_window=False)
print("QMIE window OOS:", qmie["oos"])
qmie["per_symbol"]


In [ ]:
import pandas as pd
rows = [
    {"window": "paper_2022_2024", **{k: paper["oos"][k] for k in ("sharpe", "max_dd", "cagr")}},
    {"window": "qmie_2023_now", **{k: qmie["oos"][k] for k in ("sharpe", "max_dd", "cagr")}},
]
display(pd.DataFrame(rows))


## CLI

```bash
cd python
python -m research.trend_lab.run_adaptivetrend_validation --paper-window
```

Artifact: ``research/artifacts/adaptivetrend_validation.json``. Full paper needs monthly mcap + grid — not implemented here.
